In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split, KFold
from sklearn.base import clone
from sklearn.ensemble import StackingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.feature_selection import RFE
from sklearn.utils import resample  # For simple oversampling (no imblearn needed)
from pathlib import Path
import pickle
import xgboost as xgb
import matplotlib.pyplot as plt
from scipy.stats import linregress

In [ ]:
# Load full training data
df_train = pd.read_csv("data_results/train_Data.csv", index_col=0)  # Adjust path if needed

In [ ]:
# Define core features from dataset
data_columns = df_train.columns
FEATURES = list(data_columns[:-2])
FEATURES

In [ ]:
# Surface tension integration
SIGMA = 0.072  # N/m (air-water, 20°C)
G = 9.81  # m/s²
RHO_L = 1000  # kg/m³
MU_L = 0.001  # Pa·s
U_L = 0.05  # m/s
U_B = 0.25  # m/s
Q_FLUX = 1e5  # W/m²
DELTA_T = 10  # K

In [ ]:
# Feature Engineering: Add physics-based proxies
df_train['vol_proxy'] = df_train['d_eq'] ** 3
df_train['sa_proxy'] = np.pi * df_train['d_eq'] ** 2
FEATURES_ext = FEATURES + ['vol_proxy', 'sa_proxy']

# Targets and log-transform (handles skewness)
y_v = df_train['Volume'].values
y_as = df_train['Surface_Area'].values
y_v_log = np.log1p(y_v)
y_as_log = np.log1p(y_as)

print(f"Training Dataset shape: {df_train.shape}")

# Raw feature matrix
X_raw = df_train[FEATURES_ext].values

In [ ]:
selector_v = RFE(LinearRegression(), n_features_to_select=8)
selector_v.fit(X_raw, y_v)
top_feats_v = [FEATURES_ext[i] for i in range(len(FEATURES_ext)) if selector_v.support_[i]]
X_raw_v = df_train[top_feats_v].values
scaler_v = StandardScaler()
X_v = scaler_v.fit_transform(X_raw_v)

selector_as = RFE(LinearRegression(), n_features_to_select=8)
selector_as.fit(X_raw, y_as)
top_feats_as = [FEATURES_ext[i] for i in range(len(FEATURES_ext)) if selector_as.support_[i]]
X_raw_as = df_train[top_feats_as].values
scaler_as = StandardScaler()
X_as = scaler_as.fit_transform(X_raw_as)

print("Top features Volume:", top_feats_v)
print("Top features Surface_Area:", top_feats_as)

In [ ]:
def random_oversample(X, y, target_fraction=0.5, random_state=42):
    np.random.seed(random_state)
    # Identify low-volume samples (bottom 50% quantile)
    low_idx = np.where(y < np.quantile(y, target_fraction))[0]
    X_low = X[low_idx]
    y_low = y[low_idx]
    # Oversample lows to match full size
    X_res_low, y_res_low = resample(X_low, y_low, replace=True, n_samples=len(X), random_state=random_state)
    # Combine with originals
    X_res = np.vstack([X, X_res_low])
    y_res = np.hstack([y, y_res_low])
    return X_res, y_res

X_v_res, y_v_log_res = random_oversample(X_v, y_v_log)
X_as_res, y_as_log_res = random_oversample(X_as, y_as_log)

# Polynomial features
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_v_poly = poly.fit_transform(X_v_res)
X_as_poly = poly.fit_transform(X_as_res)

print(f"Oversampled shapes: Volume {X_v_poly.shape}, SA {X_as_poly.shape}")

In [ ]:
base_estimators = [
    ('lr', LinearRegression()),
    ('rf', RandomForestRegressor(random_state=42)),
    ('gb', GradientBoostingRegressor(random_state=42)),
    ('svr', SVR()),
    ('xgb', xgb.XGBRegressor(objective='reg:squarederror', random_state=42))
]

tuning_params = {
    'lr': {},
    'rf': {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [6, 8, 12, 15, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'gb': {
        'n_estimators': [100, 200, 300, 500],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7, 9],
        'subsample': [0.8, 0.9, 1.0],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    },
    'svr': {
        'C': [100, 500, 1000],
        'gamma': ['scale', 'auto', 0.01, 0.1],
        'epsilon': [0.01, 0.05, 0.1],
        'kernel': ['rbf', 'poly']
    },
    'xgb': {
        'n_estimators': [100, 200, 300, 500],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7, 9],
        'subsample': [0.8, 0.9, 1.0],
        'colsample_bytree': [0.8, 0.9, 1.0],
        'reg_alpha': [0.01, 0.1],
        'reg_lambda': [0.1, 1.0]
    }
}

In [ ]:
best_bases_v = {}
for name, est_base in base_estimators:
    if name == 'lr':
        best_bases_v[name] = est_base
    else:
        search = RandomizedSearchCV(est_base, tuning_params[name], n_iter=50, cv=5, random_state=42, scoring='neg_mean_squared_error')
        search.fit(X_v_poly, y_v_log_res)
        best_bases_v[name] = search.best_estimator_
        print(f"Best params for {name} (Volume): {search.best_params_}")

best_bases_as = {}
for name, est_base in base_estimators:
    if name == 'lr':
        best_bases_as[name] = est_base
    else:
        search = RandomizedSearchCV(est_base, tuning_params[name], n_iter=50, cv=5, random_state=42, scoring='neg_mean_squared_error')
        search.fit(X_as_poly, y_as_log_res)
        best_bases_as[name] = search.best_estimator_
        print(f"Best params for {name} (Surface_Area): {search.best_params_}")

In [ ]:
ensemble_v = StackingRegressor(estimators=[(name, best_bases_v[name]) for name in best_bases_v], final_estimator=LinearRegression(), cv=5)
ensemble_v.fit(X_v_poly, y_v_log_res)
ensemble_v.weights_ = [0.1, 0.15, 0.15, 0.1, 0.5]  

ensemble_as = StackingRegressor(estimators=[(name, best_bases_as[name]) for name in best_bases_as], final_estimator=LinearRegression(), cv=5)
ensemble_as.fit(X_as_poly, y_as_log_res)
ensemble_as.weights_ = [0.1, 0.15, 0.15, 0.1, 0.5]

In [ ]:
def manual_cv_single(base_estimators, X, y, n_splits=5, random_state=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    ensemble_r2, ensemble_mae = [], []
    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        X_train_poly = poly.fit_transform(X_train)
        X_test_poly = poly.transform(X_test)
        y_pred_ens = np.mean([clone(est).fit(X_train_poly, y_train).predict(X_test_poly) for _, est in base_estimators], axis=0)
        ensemble_r2.append(r2_score(y_test, y_pred_ens))
        ensemble_mae.append(mean_absolute_error(y_test, y_pred_ens))
    return {'ensemble': {'R2': np.array(ensemble_r2), 'MAE': np.array(ensemble_mae)}}

cv_results_v_log = manual_cv_single([(name, best_bases_v[name]) for name in best_bases_v], X_v, y_v_log)
cv_results_as_log = manual_cv_single([(name, best_bases_as[name]) for name in best_bases_as], X_as, y_as_log)

def print_metrics(results, target_name):
    print(f"\n=== TUNED METRICS FOR {target_name} (Log Scale) ===")
    print(f"{'Model':<10} {'R² mean ± std':<15} {'MAE mean ± std':<15}")
    print("-" * 45)
    for model, scores in results.items():
        r2_mean, r2_std = scores['R2'].mean(), scores['R2'].std()
        mae_mean, mae_std = scores['MAE'].mean(), scores['MAE'].std()
        print(f"{model:<10} {r2_mean:8.3f} ± {r2_std:6.3f}  {mae_mean:8.3f} ± {mae_std:6.3f}")

print_metrics(cv_results_v_log, 'Volume')
print_metrics(cv_results_as_log, 'Surface_Area')

In [ ]:
# Hold-out on original (poly-transformed for test)
X_v_train, X_v_test, y_v_train_log, y_v_test_log = train_test_split(X_v, y_v_log, test_size=0.2, random_state=42)
X_v_train_poly = poly.fit_transform(X_v_train)
ensemble_v.fit(X_v_train_poly, y_v_train_log)
X_v_test_poly = poly.transform(X_v_test)
y_v_pred_log = ensemble_v.predict(X_v_test_poly)
y_v_test_orig = np.expm1(y_v_test_log)
y_v_pred_orig = np.expm1(y_v_pred_log)
test_r2_v = r2_score(y_v_test_orig, y_v_pred_orig)
test_mae_v = mean_absolute_error(y_v_test_orig, y_v_pred_orig)

X_as_train, X_as_test, y_as_train_log, y_as_test_log = train_test_split(X_as, y_as_log, test_size=0.2, random_state=42)
X_as_train_poly = poly.fit_transform(X_as_train)
ensemble_as.fit(X_as_train_poly, y_as_train_log)
X_as_test_poly = poly.transform(X_as_test)
y_as_pred_log = ensemble_as.predict(X_as_test_poly)
y_as_test_orig = np.expm1(y_as_test_log)
y_as_pred_orig = np.expm1(y_as_pred_log)
test_r2_as = r2_score(y_as_test_orig, y_as_pred_orig)
test_mae_as = mean_absolute_error(y_as_test_orig, y_as_pred_orig)

print(f"\nTest R² Volume (Original): {test_r2_v:.3f}, MAE: {test_mae_v:.0f}")
print(f"Test R² Surface_Area (Original): {test_r2_as:.3f}, MAE: {test_mae_as:.0f}")

In [ ]:
# Residuals
# Update the global default parameters
plt.rcParams.update({'font.size': 28, 'font.family': 'Times New Roman'})
plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
plt.scatter(y_v_test_orig, y_v_test_orig - y_v_pred_orig)
plt.xlabel('Actual Volume', fontsize=40); plt.ylabel('Residuals', fontsize=40) #; plt.title('Volume Residuals', fontsize=34)
plt.grid(True, linestyle='--', alpha=0.3)
plt.subplot(1, 2, 2)
plt.scatter(y_as_test_orig, y_as_test_orig - y_as_pred_orig)
plt.xlabel('Actual SA', fontsize=40); plt.ylabel('Residuals', fontsize=40) #; plt.title('SA Residuals', fontsize=34)
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('residuals.png')
plt.show()

In [ ]:
Path("models").mkdir(exist_ok=True)
with open("models/scaler_v.pkl", "wb") as f: pickle.dump(scaler_v, f)
with open("models/scaler_as.pkl", "wb") as f: pickle.dump(scaler_as, f)
with open("models/poly.pkl", "wb") as f: pickle.dump(poly, f)
with open("models/top_feats_v.pkl", "wb") as f: pickle.dump(top_feats_v, f)
with open("models/top_feats_as.pkl", "wb") as f: pickle.dump(top_feats_as, f)
with open("models/ensemble_v_xgb.pkl", "wb") as f: pickle.dump(ensemble_v, f)
with open("models/ensemble_as_xgb.pkl", "wb") as f: pickle.dump(ensemble_as, f)

In [ ]:
def predict_from_features(feat_dict, target='v'):
    if target == 'v':
        feats = top_feats_v; model = ensemble_v; scaler_target = scaler_v
    else:
        feats = top_feats_as; model = ensemble_as; scaler_target = scaler_as
    
    x_df = pd.DataFrame([feat_dict])[feats].fillna(0)
    x_sc = scaler_target.transform(x_df.values)
    x_poly = poly.transform(x_sc)
    pred_log = model.predict(x_poly)[0]
    ml_pred = np.expm1(pred_log)
    
    d_eq = feat_dict.get('d_eq', 0)
    ar = feat_dict.get('AR', 1.0)
    phys_weight = 0.1 / ar
    if target == 'v':
        phys_pred = (np.pi / 6) * d_eq ** 3
    else:
        phys_pred = 4 * np.pi * (d_eq / 2) ** 2
    
    final_pred = (1 - phys_weight) * ml_pred + phys_weight * phys_pred
    return final_pred

In [ ]:
# Load test data and run predictions
test_df = pd.read_csv("data_results/test_Data.csv", index_col=0)
test_df['vol_proxy'] = test_df['d_eq'] ** 3
test_df['sa_proxy'] = np.pi * test_df['d_eq'] ** 2

accuracy_data = []
for i in range(len(test_df)):
    sample_dict = test_df[FEATURES_ext].iloc[i].to_dict()
    actual_v = test_df['Volume'].iloc[i]
    pred_v = predict_from_features(sample_dict, 'v')
    mape_v = abs(actual_v - pred_v) / actual_v * 100 if actual_v != 0 else 0
    actual_as = test_df['Surface_Area'].iloc[i]
    pred_as = predict_from_features(sample_dict, 'as')
    mape_as = abs(actual_as - pred_as) / actual_as * 100 if actual_as != 0 else 0
    accuracy_data.append({
        "Actual Volume": actual_v, "Prediction Volume": pred_v, "MAPE Volume": mape_v,
        "Actual Surface Area": actual_as, "Predicted Surface Area": pred_as, "MAPE Surface Area": mape_as
    })

Prediction_Results = pd.DataFrame(accuracy_data)
print(Prediction_Results.describe())
Prediction_Results.to_csv("data_results/prediction_Results.csv", index=False)

In [ ]:
Prediction_Results.head()

Error Graphs

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import gaussian_kde
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
figsize = (12, 8)

In [ ]:
# Extract predictions from your loop
actual_v = Prediction_Results['Actual Volume'].values
pred_v = Prediction_Results['Prediction Volume'].values
actual_sa = Prediction_Results['Actual Surface Area'].values
pred_sa = Prediction_Results['Predicted Surface Area'].values
mape_v = Prediction_Results['MAPE Volume'].values
mape_sa = Prediction_Results['MAPE Surface Area'].values

In [ ]:
# For hold-out test set (from earlier split)
y_v_test_true = y_v_test_orig
y_v_pred_final = np.expm1(ensemble_v.predict(X_v_test_poly))  # ML-only
# Hybrid on test set (using d_eq and AR if available, fallback to ML-only)
try:
    ar_test = test_df['AR'].values[:len(y_v_test_true)] if 'AR' in test_df.columns else np.ones_like(y_v_test_true)
    d_eq_test = test_df['d_eq'].values[:len(y_v_test_true)] if 'd_eq' in test_df.columns else np.ones_like(y_v_test_true)
    phys_weight_v = np.clip(0.1 / np.maximum(ar_test, 1.0), 0, 0.3)
    phys_pred_v = (np.pi/6) * d_eq_test**3
    y_v_pred_hybrid = (1 - phys_weight_v) * y_v_pred_final + phys_weight_v * phys_pred_v
except:
    y_v_pred_hybrid = y_v_pred_final

In [ ]:
# Update the global default parameters
plt.rcParams.update({'font.size': 28, 'font.family': 'Times New Roman'})
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
min_v, max_v = min(actual_v.min(), pred_v.min()), max(actual_v.max(), pred_v.max())
min_sa, max_sa = min(actual_sa.min(), pred_sa.min()), max(actual_sa.max(), pred_sa.max())

ax1.scatter(actual_v, pred_v, s=150, color='green', label='Final Hybrid Prediction')
ax1.plot([min_v, max_v], [min_v, max_v], 'r--', lw=2, label='Perfect', alpha=0.9)
ax1.spines['bottom'].set_edgecolor('black')
ax1.spines['left'].set_edgecolor('black')
ax1.spines['top'].set_edgecolor('black')
ax1.spines['right'].set_edgecolor('black')
ax1.spines['bottom'].set_linewidth(2)
ax1.spines['left'].set_linewidth(2)
ax1.spines['top'].set_linewidth(2)
ax1.spines['right'].set_linewidth(2)
ax1.set_xlabel('Actual Volume (mm³)', fontsize=34); ax1.set_ylabel('Predicted Volume (mm³)', fontsize=34)
ax1.grid(True, linestyle='--')
# ax1.set_title(f'Volume Prediction\nR² = {r2_score(actual_v, pred_v):.3f} | MAE = {mean_absolute_error(actual_v, pred_v):.1f}')
ax1.legend(); #ax1.axis('equal')

ax2.scatter(actual_sa, pred_sa, s=150, color='blue', label='Final Hybrid Prediction')
ax2.plot([min_sa, max_sa], [min_sa, max_sa], 'r--', lw=2, label='Perfect')
ax2.spines['bottom'].set_edgecolor('black')
ax2.spines['left'].set_edgecolor('black')
ax2.spines['top'].set_edgecolor('black')
ax2.spines['right'].set_edgecolor('black')
ax2.spines['bottom'].set_linewidth(2)
ax2.spines['left'].set_linewidth(2)
ax2.spines['top'].set_linewidth(2)
ax2.spines['right'].set_linewidth(2)
ax2.set_xlabel('Actual Surface Area (mm²)', fontsize=34); ax2.set_ylabel('Predicted Surface Area (mm²)', fontsize=34)
ax2.grid(True, linestyle='--')
# ax2.set_title(f'Surface Area Prediction\nR² = {r2_score(actual_sa, pred_sa):.3f} | MAE = {mean_absolute_error(actual_sa, pred_sa):.1f}')
ax2.legend(); #ax2.axis('equal')

# plt.suptitle('Actual vs Predicted (Final Test Set with Physics Hybrid)', fontsize=16)
plt.tight_layout()
plt.savefig("1_actual_vs_predicted.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Update the global default parameters
plt.rcParams.update({'font.size': 28, 'font.family': 'Times New Roman'})
plt.figure(figsize=(12, 8))
sns.histplot(mape_v, bins=50, kde=True, label=f'Volume (median {np.median(mape_v):.1f}%)', color='blue')
sns.histplot(mape_sa, bins=50, kde=True, label=f'Surface Area (median {np.median(mape_sa):.1f}%)', color='orange')
plt.axvline(np.median(mape_v), color='blue', linestyle='--')
plt.axvline(np.median(mape_sa), color='orange', linestyle='--')
plt.xlabel('MAPE (%)', fontsize=32)
plt.ylabel('Count', fontsize=32)
# plt.title('Distribution of Mean Absolute Percentage Error (MAPE)')
plt.legend()
ax = plt.gca()
plt.setp(ax.spines.values(), color='black', linewidth=2)
plt.xlim(0, 30)
plt.grid(True, linestyle='--')
# plt.tight_layout()
plt.savefig("2_mape_distribution.png", dpi=300)
plt.show()

In [ ]:
# Update the global default parameters
plt.rcParams.update({'font.size': 28, 'font.family': 'Times New Roman'})
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
ax1.scatter(pred_v, actual_v - pred_v, color='green', s=150)
ax1.axhline(0, color='red', linestyle='--')
ax1.spines['bottom'].set_edgecolor('black')
ax1.spines['left'].set_edgecolor('black')
ax1.spines['top'].set_edgecolor('black')
ax1.spines['right'].set_edgecolor('black')
ax1.spines['bottom'].set_linewidth(2)
ax1.spines['left'].set_linewidth(2)
ax1.spines['top'].set_linewidth(2)
ax1.spines['right'].set_linewidth(2)
ax1.set_xlabel('Predicted Volume', fontsize=34); ax1.set_ylabel('Residual', fontsize=34)
ax1.grid(True, linestyle='--')
# ax1.set_title('Residuals vs Predicted - Volume')

ax2.scatter(pred_sa, actual_sa - pred_sa, color='blue', s=150)
ax2.axhline(0, color='red', linestyle='--')
ax2.spines['bottom'].set_edgecolor('black')
ax2.spines['left'].set_edgecolor('black')
ax2.spines['top'].set_edgecolor('black')
ax2.spines['right'].set_edgecolor('black')
ax2.spines['bottom'].set_linewidth(2)
ax2.spines['left'].set_linewidth(2)
ax2.spines['top'].set_linewidth(2)
ax2.spines['right'].set_linewidth(2)
ax2.set_xlabel('Predicted Surface Area', fontsize=34);  ax2.set_ylabel('Residual', fontsize=34)
ax2.grid(True, linestyle='--')
# ax2.set_title('Residuals vs Predicted - Surface Area')
plt.tight_layout()
plt.savefig("3_residuals_vs_predicted.png", dpi=300)
plt.show()

In [ ]:
def plot_error_grid(y_true, y_pred, title, ax, color='Reds'):
    lim = [max(1, y_true.min()*0.9), y_true.max()*1.1]
    bins = np.logspace(np.log10(lim[0]), np.log10(lim[1]), 20)
    h, xedges, yedges = np.histogram2d(y_true, y_pred, bins=(bins, bins))
    X, Y = np.meshgrid(xedges, yedges)
    im = ax.pcolormesh(X, Y, h.T, cmap=color)
    ax.plot(lim, lim, 'r--', lw=2)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('Predicted', fontsize=34); ax.set_ylabel('Actual', fontsize=34)
    ax.spines['bottom'].set_edgecolor('black')
    ax.spines['left'].set_edgecolor('black')
    ax.spines['top'].set_edgecolor('black')
    ax.spines['right'].set_edgecolor('black')
    ax.spines['bottom'].set_linewidth(2)
    ax.spines['left'].set_linewidth(2)
    ax.spines['top'].set_linewidth(2)
    ax.spines['right'].set_linewidth(2)
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.axis('equal')
    # ax.set_title(title, fontsize=34)
    plt.colorbar(im, ax=ax, label='Count')

# Update the global default parameters
plt.rcParams.update({'font.size': 22, 'font.family': 'Times New Roman'})
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
plot_error_grid(actual_v, pred_v, 'Error Grid - Volume', ax1)
plot_error_grid(actual_sa, pred_sa, 'Error Grid - Surface Area', ax2)
# plt.suptitle('Regression "Confusion Matrix" (Log-Scale Bins)', fontsize=16)
plt.tight_layout()
plt.savefig("4_error_grid.png", dpi=300)
plt.show()

In [ ]:
try:
    # Update the global default parameters
    plt.rcParams.update({'font.size': 28, 'font.family': 'Times New Roman'})
    ar_values = test_df['AR'].values
    plt.figure(figsize=(12, 8))
    plt.scatter(ar_values, mape_v, label='Volume MAPE', s=150, color="red")
    z = np.polyfit(ar_values, mape_v, 1)
    p = np.poly1d(z)
    plt.plot(ar_values, p(ar_values), "b--", lw=2, label=f'Trend (slope={z[0]:.2f})')
    plt.xlabel('Aspect Ratio (AR)', fontsize=30)
    plt.ylabel('MAPE Volume (%)', fontsize=30)
    # plt.title('Does Physics Hybrid Help Elongated Bubbles? (Lower MAPE = Better)')
    plt.legend()
    ax = plt.gca()
    plt.setp(ax.spines.values(), color='black', linewidth=2)
    plt.grid(True, linestyle='--')
    # plt.tight_layout()
    plt.savefig("5_bias_by_aspect_ratio.png", dpi=300)
    plt.show()
except:
    print("AR column not found - skipping aspect ratio bias plot")

In [ ]:
summary = pd.DataFrame({
    'Target': ['Volume', 'Surface Area'],
    'R²': [r2_score(actual_v, pred_v), r2_score(actual_sa, pred_sa)],
    'MAE': [mean_absolute_error(actual_v, pred_v), mean_absolute_error(actual_sa, pred_sa)],
    'MAPE (%)': [np.median(mape_v), np.median(mape_sa)],
    'Max MAPE (%)': [mape_v.max(), mape_sa.max()],
    'Samples': [len(actual_v), len(actual_sa)]
})
print("\n" + "="*60)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("="*60)
print(summary.round(3).to_string(index=False))
summary.to_csv("model_performance_summary.csv", index=False)

# Save final results with diagnostics
Prediction_Results['Relative_Error_V_%'] = mape_v
Prediction_Results['Relative_Error_SA_%'] = mape_sa
# Prediction_Results.to_csv("FINAL_Prediction_Results_with_Diagnostics.csv", index=False)